In [1]:
import sys
import os
sys.path.append(os.path.abspath('/acne-lds/model'))
sys.path.append(os.path.abspath('model'))
sys.path.append(os.path.abspath("/acne-lds/utils"))

In [2]:
from model_ld_smoothing import AcneModel

/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/io/image.py:11: UserWarning: Failed to load image Python extension: dlopen(/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libpng16.16.dylib
  Referenced from: <5F6B6919-410D-397C-98F2-12C5934F9DBE> /opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/image.so
  Reason: tried: '/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/System/Volu

In [163]:
from predict_on_img import ModelInit
from PIL import Image

model = ModelInit(path_checkpoint='model/lds-weights/model_fold_4.pth')
# img = Image.open(PATH_TO_IMAGE)
# predictions = model.predict_on_img(img)

In [159]:
# Assume ModelInit is imported and your checkpoint path is correct
import torch
# model_wrapper = ModelInit(model_type="model_ld_smoothing", path_checkpoint="model/lds-weights/model_fold_4.pth", device="cpu")
# model = model_wrapper.model
# model.eval()

# Dummy input tensor (use actual input size expected, e.g. 3x224x224)
example_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(model, example_input)

In [157]:
import torch
from torch import nn
from collections import namedtuple

# Define a namedtuple for outputs
ModelOutput = namedtuple("ModelOutput", ["cls_pred", "lesion_count"])

class ModelWrapper(nn.Module):
    def __init__(self, model, model_type="model_ld_smoothing"):
        super().__init__()
        self.model = model
        self.model_type = model_type

    def forward(self, x):
        return self.model.predict_on_img(x)
        # cls, cou, cou2cls = self.model(x)

        # if self.model_type == "model_ld_smoothing":
        #     cls = torch.stack(
        #         (
        #             torch.sum(cls[:, :1], 1),
        #             torch.sum(cls[:, 1:4], 1),
        #             torch.sum(cls[:, 4:10], 1),
        #             torch.sum(cls[:, 10:], 1),
        #         ),
        #         dim=1,
        #     )

        # cls_pred = torch.argmax(0.5 * (cls + cou2cls), dim=1)
        # lesion_count = torch.argmax(cou, dim=1) + 1

        # return ModelOutput(cls_pred, lesion_count)

# Wrap your model
# Initialize model (make sure the path is correct)
model_wrapper = ModelInit(model_type="model_ld_smoothing", path_checkpoint="model/lds-weights/model_fold_4.pth", device="cpu")
model = model_wrapper.model
model.eval()

wrapped_model = ModelWrapper(model)
# wrapped_model.eval()
# Then trace the wrapped model

traced_model = torch.jit.trace(wrapped_model, example_input)

RuntimeError: Tracer cannot infer type of (array([[-1.28973089,  0.58097548,  0.5712603 , ..., -0.64180039,
         0.46786874,  1.06329637],
       [ 1.00733353,  1.14818674,  1.24522454, ...,  1.33040123,
        -0.43101617, -1.22685803],
       [ 0.42915429,  0.38813159,  0.49125526, ..., -0.7235787 ,
         1.37276515,  0.34595705],
       ...,
       [-0.39267356,  0.5352046 , -1.96071506, ..., -1.01418642,
         1.42742573,  1.5710015 ],
       [-3.17906772, -0.94473221, -0.09914311, ..., -2.00256112,
        -0.40568627,  0.38362243],
       [-0.57643268, -0.05840524, -0.54435947, ...,  0.58335442,
        -0.19450204,  0.69489501]]), array([[-0.29853804, -0.29509146,  0.92914357, ...,  0.23700755,
         0.27246625,  0.71397153],
       [ 0.15434154, -0.46005345,  0.82359357, ...,  0.20585802,
         0.06948632, -0.5428737 ],
       [ 0.0852203 , -0.47415531,  0.96351163, ...,  0.21793754,
         1.40865819,  0.37178167],
       ...,
       [ 0.06578054,  0.9231277

In [145]:
import coremltools as ct
coreml_model = ct.convert(
    traced_model,
    convert_to="neuralnetwork",
    inputs=[ct.ImageType(
        name="input_image",
        shape=(1, 3, 224, 224),
        bias=[-0.45815152, -0.361242, -0.29348266],
        scale=1/0.20132513,  # Fixed: all three channel scales
        color_layout=ct.colorlayout.RGB,

    )]
)

Tuple detected at graph output. This will be flattened in the converted model.
Running MIL default pipeline:   0%|          | 0/87 [00:00<?, ? passes/s]/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '854', of the source model, has been renamed to 'var_854' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '923', of the source model, has been renamed to 'var_923' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Translating MIL ==> NeuralNetwork Ops: 100%|██████████| 544/544 [00:17<00:00, 31.05 ops/s] 


In [130]:
coreml_model_lut = ct.models.neural_network.quantization_utils.quantize_weights(
    coreml_model,
    nbits=8, 
    quantization_mode="linear_lut"
)

Quantizing using linear_lut quantization
Optimizing Neural Network before Quantization:
Finished optimizing network. Quantizing neural network..
Quantizing layer input.3 of type convolution
Quantizing layer input.11 of type convolution
Quantizing layer input.17 of type convolution
Quantizing layer out.1 of type convolution
Quantizing layer residual.1 of type convolution
Quantizing layer input.31 of type convolution
Quantizing layer input.37 of type convolution
Quantizing layer out.3 of type convolution
Quantizing layer input.49 of type convolution
Quantizing layer input.55 of type convolution
Quantizing layer out.5 of type convolution
Quantizing layer input.67 of type convolution
Quantizing layer input.73 of type convolution
Quantizing layer out.7 of type convolution
Quantizing layer residual.3 of type convolution
Quantizing layer input.87 of type convolution
Quantizing layer input.93 of type convolution
Quantizing layer out.9 of type convolution
Quantizing layer input.105 of type conv

In [119]:
coreml_model_lut.save("AcneClassQuantImpR.mlpackage")

In [131]:
img = Image.open('../data/Classification/JPEGImages/levle1_222.jpg')

In [132]:
from PIL import Image
import numpy as np
img_resized = img.resize((224, 224))

In [160]:
outputs = coreml_model.predict({"input_image": img_resized})



In [161]:
outputs

{'var_923': array([[1.        , 0.        , 0.00300217, 0.00150108]], dtype=float32),
 'cou': array([[1.00016594e-04, 1.00000000e+00, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.00016594e-04, 1.00016594e-04, 1.00016594e-04,
         1.00016594e-04, 1.0001

In [137]:
coreml_model_lut.predict({"input_image": img_resized})

{'var_916': array([2.], dtype=float32), 'var_910': array([0.], dtype=float32)}

In [167]:
cls, cou, cou2cls = model.predict_on_img(img)

In [168]:
cls

tensor([[1.3652e-04, 1.9920e-01, 8.0167e-01, 3.0086e-04]])

In [165]:
torch.argmax(0.5 * (cls + cou2cls), dim=1), torch.argmax(cou, dim=1) + 1

(tensor([2]), tensor([25]))